# Train a neural signed-distance field — DeepSDF-style

**Track B · 3D & Neural Rendering** · maps to lesson B4 (implicit surfaces).

We fit an MLP `f(x,y,z) → signed distance` to a target shape, then extract the surface as the zero level set with marching cubes. Pure PyTorch; the target here is analytic (a torus) so the notebook is fully self-contained — swap in your own point samples to fit any mesh.

> CPU is fine for this lab; a GPU makes it instant.

In [ ]:
import os, torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 2000))
print("device:", device, "| steps:", STEPS)

## 1 · Target shape — the true SDF of a torus
Negative inside, zero on the surface, positive outside.

In [ ]:
def torus_sdf(p, R=0.6, r=0.22):
    xz = torch.linalg.norm(p[..., [0, 2]], dim=-1) - R
    q = torch.stack([xz, p[..., 1]], dim=-1)
    return torch.linalg.norm(q, dim=-1) - r

def sample(n):
    # half uniform in the box, half concentrated near the surface so the
    # zero-crossing is well supervised
    nu = n // 2
    pu = (torch.rand(nu, 3, device=device) * 2 - 1) * 1.1
    a = torch.rand(n - nu, device=device) * 2 * np.pi
    b = torch.rand(n - nu, device=device) * 2 * np.pi
    R, r = 0.6, 0.22
    ps = torch.stack([(R + r * torch.cos(b)) * torch.cos(a),
                      r * torch.sin(b),
                      (R + r * torch.cos(b)) * torch.sin(a)], -1)
    ps = ps + 0.05 * torch.randn(n - nu, 3, device=device)
    p = torch.cat([pu, ps])
    return p, torus_sdf(p)

## 2 · Model — a positional-encoded MLP regressing distance

In [ ]:
def enc(x, L=4):
    out = [x]
    for i in range(L):
        for fn in (torch.sin, torch.cos):
            out.append(fn(2.0 ** i * x))
    return torch.cat(out, -1)

class SDF(nn.Module):
    def __init__(self, L=4, Wd=256):
        super().__init__(); self.L = L; din = 3 * (1 + 2 * L)
        self.net = nn.Sequential(
            nn.Linear(din, Wd), nn.Softplus(beta=100),
            nn.Linear(Wd, Wd), nn.Softplus(beta=100),
            nn.Linear(Wd, Wd), nn.Softplus(beta=100),
            nn.Linear(Wd, 1))
    def forward(self, x):
        return self.net(enc(x, self.L)).squeeze(-1)

## 3 · Train — regress the (target-clamped) distance, the DeepSDF loss
We clamp the *target* to ±0.1 to focus capacity near the surface. Note we do **not** clamp the prediction — clamping it would zero out gradients whenever the output drifts past ±0.1 and the model would never learn.

In [ ]:
model = SDF().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3)
hist = []
for step in range(STEPS + 1):
    p, gt = sample(4096)
    pred = model(p)
    loss = (pred - gt.clamp(-0.1, 0.1)).abs().mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        hist.append((step, loss.item())); print(f"step {step:5d}  L1 {loss.item():.4f}")

## 4 · Extract & compare the surface (marching cubes)

In [ ]:
from skimage import measure
res = 96
g = torch.linspace(-1.1, 1.1, res, device=device)
gx, gy, gz = torch.meshgrid(g, g, g, indexing="ij")
grid = torch.stack([gx, gy, gz], -1).reshape(-1, 3)
with torch.no_grad():
    vol = torch.cat([model(grid[i:i+50000]) for i in range(0, grid.shape[0], 50000)]).reshape(res, res, res).cpu().numpy()
verts, faces, _, _ = measure.marching_cubes(vol, level=0.0)
fig = plt.figure(figsize=(9, 4))
ax1 = fig.add_subplot(121, projection="3d")
ax1.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2], triangles=faces, cmap="viridis", lw=0)
ax1.set_title("learned surface (f=0)"); ax1.set_axis_off()
ax2 = fig.add_subplot(122); ax2.plot(*zip(*hist), "-o"); ax2.set_title("training L1 ↓"); ax2.set_xlabel("step"); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/B_deepsdf_shape/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/B_deepsdf_shape"; os.makedirs(run, exist_ok=True)
torch.save(model.state_dict(), f"{run}/sdf.pt")
json.dump({"l1": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## How this links to tracks A–D
Implicit surfaces are a **D · Scene / world** map substrate.

### Where to go next
- Replace `torus_sdf` with on/near-surface samples from your own mesh (e.g. via `trimesh`) to fit arbitrary shapes.
- Add an **eikonal** term `(||∇f|| − 1)²` for a metric SDF → **IGR / SIREN / NeuS**.
- Condition the MLP on a per-shape latent code → the full **DeepSDF** auto-decoder.